In [113]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm


In [114]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(720, 600, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

In [115]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [116]:
fragment_code = """
        uniform vec4 color;
        uniform int night;
        void main(){
            if(night == 0){
                gl_FragColor = color;
            }
            else{
                gl_FragColor = 0.5*color;
            }
            
        }
        """

In [117]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compile shaders
glCompileShader(vertex)

if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)




In [118]:
# preparando espaço para 3 vértices usando 2 coordenadas (x,y)
vertices = np.zeros(0, [("position", np.float32, 3)])

def make_sphereindices(numv, numh):
    indices = []

    for i in range(numv):
        for j in range(numh):
            p0 = i * (numh + 1) + j
            p1 = p0 + 1
            p2 = p0 + (numh + 1)
            p3 = p2 + 1

            # Triângulo 1
            indices.append((p0, p2, p1))

            # Triângulo 2
            indices.append((p1, p2, p3))

    return np.array(indices, dtype=np.uint32)

def makesphere(numv, numh, radius, bump):
    spherevertices = []

    for i in range(numv + 1):
        phi = i * np.pi / numv  

        for j in range(numh + 1):
            theta = j * 2 * np.pi / numh  

            x = np.sin(phi) * np.cos(theta) * radius + (np.random.randint(-10, 10) * 0.01 *bump)
            y = np.sin(phi) * np.sin(theta) * radius + (np.random.randint(-10, 10) * 0.01 *bump)
            z = np.cos(phi) * radius  

            spherevertices.append((x, y, z))

    return np.array(spherevertices, dtype=np.float32)

def makecircle(num_vertices, radius):
    circvertices = np.zeros(num_vertices, [("position", np.float32, 3)])

    angle = 0.0
    for counter in range(num_vertices):
        angle += 2*np.pi/num_vertices 
        x = np.cos(angle)*radius
        y = np.sin(angle)*radius
        circvertices[counter]['position'] = [x,y,0.0]
    return circvertices

def Make_sol():
    global solStart, solLen, vertices

    solVertices = []
    solStart = len(vertices['position'])

    spherevertices = makesphere(20, 20, 0.12, 0)
    sphereindices = make_sphereindices(20, 20)

    spike_length = 0.05  

    counter = 0
    for index in sphereindices:
        v0 = spherevertices[index[0]]
        v1 = spherevertices[index[1]]
        v2 = spherevertices[index[2]]

        if (counter%7 == 0):
        
            center = (v0 + v1 + v2) / 3.0

            
            normal = np.cross(v1 - v0, v2 - v0)
            normal = normal / np.linalg.norm(normal)

            
            tip = center + normal * spike_length

            solVertices.append(v0)
            solVertices.append(v1)
            solVertices.append(v2)
            
            solVertices.append(v0)
            solVertices.append(v1)
            solVertices.append(tip)

            solVertices.append(v1)
            solVertices.append(v2)
            solVertices.append(tip)

            solVertices.append(v2)
            solVertices.append(v0)
            solVertices.append(tip)
        
        else:
            solVertices.append(v0)
            solVertices.append(v1)
            solVertices.append(v2)

        counter+=1

    solVertices = np.array(solVertices, dtype=np.float32)
    solLen = len(solVertices)

    sol_structured = np.zeros(solLen, dtype=[("position", np.float32, 3)])
    sol_structured["position"] = solVertices

    vertices = np.concatenate((vertices, sol_structured))


def make_luna():
    global lunaStart, lunaLen, vertices

    lunaVertices = []
    lunaStart = len(vertices['position'])

    spherevertices = makesphere(20, 20, 0.10, 0.03)
    sphereindices = make_sphereindices(20, 20)


    for index in sphereindices:
        v0 = spherevertices[index[0]]
        v1 = spherevertices[index[1]]
        v2 = spherevertices[index[2]]

        lunaVertices.append(v0)
        lunaVertices.append(v1)
        lunaVertices.append(v2)



    lunaVertices = np.array(lunaVertices, dtype=np.float32)
    lunaLen = len(lunaVertices)

    luna_structured = np.zeros(lunaLen, dtype=[("position", np.float32, 3)])
    luna_structured["position"] = lunaVertices

    vertices = np.concatenate((vertices, luna_structured))

def Make_house():
    global houseStart, houseLen, vertices
    houseStart = len(vertices['position'])
    houseVertices = np.zeros(9, [("position", np.float32, 3)])
    houseVertices['position'] = [[0.0,0.0,0.0],
                                 [1.0,0.0,0.0],
                                 [1.0,1.0,0.0],
                                 [0.0,1.0,0.0],
                                 [1.0,1.0,0.0],
                                 [0.0,0.0,0.0],
                                 [-0.2,1.0,0.0],
                                 [1.2,1.0,0.0],
                                 [0.5,1.5,0.0]]

    houseLen = len(houseVertices['position'])
    vertices = np.concatenate((vertices, houseVertices))

def Make_Ground():
    global groundStart, groundLen, vertices
    groundStart = len(vertices['position'])
    groundVertices = np.zeros(4, [("position", np.float32, 3)])
    groundVertices['position'] = [[-5.0,0.0,0.0],
                                 [5.0,0.0,0.0],
                                 [-5.0,-5.0,0.0],
                                 [5.0,-5.0,0.0]]

    groundLen = len(groundVertices['position'])
    vertices = np.concatenate((vertices, groundVertices))

def Make_Cloud():
    global cloudStart, cloudLen, vertices
    cloudStart = len(vertices['position'])
    
    # 3 círculos para formar a nuvem
    c1 = makecircle(20, 0.10)
    # Circulo 2 para baixo e para a esquerda
    c2 = makecircle(20, 0.06)
    c2['position'][:, 0] -= 0.12
    c2['position'][:, 1] -= 0.02
    # Circulo 3 para baixo e para a direita
    c3 = makecircle(20, 0.06)
    c3['position'][:, 0] += 0.12
    c3['position'][:, 1] -= 0.02
    
    # Junta os três círculos num array só
    cloudVertices = np.concatenate((c1, c2, c3))
    
    # Todos mesmo tamanho
    cloudLen = len(c1) 
    vertices = np.concatenate((vertices, cloudVertices))
    
def Make_Tree():
    global treeStart, treeLen, vertices
    treeStart = len(vertices['position'])
    
    # Tronco: retangulo de 2 triangulos (6 vértices)
    tronco = np.zeros(6, [("position", np.float32, 3)])
    tronco['position'] = [
        [-0.05, 0.0, 0.0], [0.05, 0.0, 0.0], [0.05, 0.2, 0.0],
        [-0.05, 0.0, 0.0], [0.05, 0.2, 0.0], [-0.05, 0.2, 0.0]
    ]
    
    # Folhas: triangulo grande em cima do tronco (3 vértices)
    folhas = np.zeros(6, [("position", np.float32, 3)])
    folhas['position'] = [
        [-0.2, 0.1, 0.0],
        [ 0.2, 0.1, 0.0],
        [ 0.0, 0.4, 0.0],
        [-0.2, 0.2, 0.0],
        [ 0.2, 0.2, 0.0],
        [ 0.0, 0.5, 0.0]   
    ]
                                 
    treeVertices = np.concatenate((tronco, folhas))
    treeLen = len(treeVertices['position'])
    vertices = np.concatenate((vertices, treeVertices))

Make_sol()
make_luna()
Make_house()
Make_Ground()
Make_Cloud()
Make_Tree()
vertices

/tmp/ipykernel_21554/99317717.py:73: RuntimeWarning: invalid value encountered in divide
  normal = normal / np.linalg.norm(normal)


array([([ 0.        ,  0.        ,  0.12      ],),
       ([ 0.01877214,  0.        ,  0.1185226 ],),
       ([ 0.        ,  0.        ,  0.12      ],), ...,
       ([-0.2       ,  0.2       ,  0.        ],),
       ([ 0.2       ,  0.2       ,  0.        ],),
       ([ 0.        ,  0.5       ,  0.        ],)],
      shape=(5920,), dtype=[('position', '<f4', (3,))])

In [119]:


def drawSol():
    angle = day_cycle

    mat_rotation1       = np.array([    [np.cos(angle), 0.0, np.sin(angle), 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [-np.sin(angle), 0.0, np.cos(angle), 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)

    mat_rotation2       = np.array([    [np.cos(angle), -np.sin(angle), 0.0, 0.0], 
                                    [np.sin(angle), np.cos(angle), 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, 0.8], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_rotation1
    mat_final = mat_rotation2 @ mat_final
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 1.0, 1.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, solStart, solLen)


def drawLuna():
    angle = day_cycle

    mat_rotation1       = np.array([    [np.cos(angle), 0.0, np.sin(angle), 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [-np.sin(angle), 0.0, np.cos(angle), 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)

    mat_rotation2       = np.array([    [np.cos(angle), -np.sin(angle), 0.0, 0.0], 
                                    [np.sin(angle), np.cos(angle), 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.8], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_rotation1
    mat_final = mat_rotation2 @ mat_final
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 1.8, 1.8, 1.7, 1.0) 
    glDrawArrays(GL_TRIANGLES, lunaStart, lunaLen)



def drawHouse():

    mat_scale      = np.array([    [0.2, 0.0, 0.0, 0.0], 
                                    [0.0, 0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_scale @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.95, 0.95, 0.90, 1.0) 
    glDrawArrays(GL_TRIANGLES, houseStart, houseLen-3)

    glUniform4f(loc_color, 0.90, 0.0, 0.10, 1.0) 
    glDrawArrays(GL_TRIANGLES, houseStart+6, houseLen-6)


def drawhouseShadow():
    if night == 1.0:
        return
    
    sc = -np.cos(day_cycle)*2.0
    mat_scale      = np.array([    [-0.2, 0.0, 0.0, 0.0], 
                                    [0.0, -0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_shearing = np.array([    [1.0, -sc, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_shearing
    mat_final = mat_scale @ mat_final
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, houseStart, houseLen)


def drawGround():

    mat_final = np.array([    [1.0, 0.0, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.1, 0.75, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, groundStart, groundLen)

def drawCloud():
    if night == 1.0:
        return 
        
    # Quando day_cycle é 0, tx = 1.0
    # Quando day_cycle é pi, tx = -1.0
    tx = 1.0 - (day_cycle / np.pi)*2.0
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, tx], 
                                    [0.0, 1.0, 0.0, 0.6], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_translation)

    glUniform4f(loc_color, 1.0, 1.0, 1.0, 1.0) 
    
    # Desenha os 3 círculos da nuvem
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart, cloudLen)                  
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart + cloudLen, cloudLen)       
    glDrawArrays(GL_TRIANGLE_FAN, cloudStart + 2*cloudLen, cloudLen)

def drawTree():
    sy = 0.5 + total_time*0.2
    sx = 0.5 + total_time*0.1

    if(sy > 2.5):
        sy = 2.5
        sx = 1.5
    
    mat_scale = np.array([          [sx, 0.0, 0.0, 0.0], 
                                    [0.0, sy, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.6], 
                                    [0.0, 1.0, 0.0, -0.5], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_scale
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

    # Tronco marrom
    glUniform4f(loc_color, 0.55, 0.27, 0.07, 1.0)
    glDrawArrays(GL_TRIANGLES, treeStart, 6)

    # Folhas verdes
    glUniform4f(loc_color, 0.0, 0.6, 0.2, 1.0)
    glDrawArrays(GL_TRIANGLES, treeStart + 6, 6)


def drawtreeShadow():
    if night == 1.0:
        return
        
    sy = 0.5 + total_time*0.2
    sx = 0.5 + total_time*0.1

    if(sy > 2.5):
        sy = 2.5
        sx = 1.5
        
    sc = -np.cos(day_cycle)*2.0
    mat_scale      = np.array([    [sx, 0.0, 0.0, 0.0], 
                                    [0.0, -sy, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
        
    mat_shearing = np.array([    [1.0, -sc, 0.0, 0.0], 
                                [0.0, 1.0, 0.0, 0.0], 
                                [0.0, 0.0, 1.0, 0.0], 
                                [0.0, 0.0, 0.0, 1.0]], np.float32)
        
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.6], 
                                    [0.0, 1.0, 0.0, -0.5], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
        
    mat_final = mat_shearing @ mat_scale
    mat_final = mat_translation @ mat_final
        
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

    
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLES, treeStart, treeLen)

In [120]:
glBufferData(GL_ARRAY_BUFFER, vertices['position'].nbytes, vertices['position'], GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

loc_color = glGetUniformLocation(program, "color")

R = 0.7
G = 0.0
B = 0.2



In [121]:
# exemplo para matriz de escala

def key_event(window,key,scancode,action,mods):
    global total_time, day_cycle, night, fill, keytimer
    
    # Seta esquerda avança o tempo
    if key == 263: 
         total_time += 0.01
    # Seta direita volta o tempo
    if key == 262: 
         total_time -= 0.01

    if ((key == 80) and(keytimer > 9)):
        if(fill == 1):
            glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
            fill = 0
        else:
            glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)
            fill = 1
        keytimer = 0

    # Evita árvore escalando negativa
    if total_time < 0.0:
        total_time = 0.0

    day_cycle = total_time % (2*np.pi)

    if day_cycle > np.pi:
        night = 1.0
    else:
        night = 0.0

    loc_night = glGetUniformLocation(program, "night")
    glUniform1i(loc_night, int(night))
    
glfw.set_key_callback(window,key_event)


In [122]:
glfw.show_window(window)
total_time = 0.0
day_cycle = 0.0
night = 0.0
fill = 1
keytimer = 0
while not glfw.window_should_close(window):

    keytimer += 1
    if keytimer > 10:
        keytimer = 10


    if (night == 0.0):
        glClearColor(173.0/255, 216.0/255, 230.0/255, 1.0)
        
    else:   
        glClearColor(0.1, 0.1, 0.5, 1.0)

    glClear(GL_COLOR_BUFFER_BIT) 
    
    drawSol()
    drawLuna()
    drawCloud()
    drawGround()
    drawHouse()
    drawhouseShadow()
    drawTree()
    drawtreeShadow()
    
    glfw.swap_buffers(window)
    glfw.poll_events() 

glfw.terminate()